In [1]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# 1. Setup Environment
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["TRAINING_ENV"] = "kaggle"

# 2. Clone Repository
REPO_URL = "https://github.com/Firojpaudel/chatterbox-nepali.git"
REPO_DIR = "/kaggle/working/chatterbox-nepali"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Updating repository...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# 3. Install Dependencies
print("Installing dependencies...")
# 💡 FIX: Explicitly install matched torch/torchvision versions to avoid 'torchvision::nms' errors on Kaggle
subprocess.run(["pip", "install", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run(["pip", "install", "-e", "."], check=True)
subprocess.run(["pip", "install", "omegaconf", "conformer", "s3tokenizer", "safetensors", "pyloudnorm", "wandb", "huggingface-hub"], check=True)
subprocess.run(["pip", "install", "git+https://github.com/resemble-ai/Perth.git@master"], check=True)

print("\n--- Chatterbox Setup Complete ---")

In [2]:
%cd /kaggle/working/chatterbox-nepali

In [3]:
# Download existing checkpoint from Hugging Face
from huggingface_hub import hf_hub_download
print("Downloading t3_nepali_epoch_20.pt...")
ckpt_path = hf_hub_download(repo_id="officialuser/chatterbox-nepali", filename="t3_nepali_epoch_20.pt")
dest = "t3_nepali_epoch_20.pt"
if os.path.exists(dest): os.remove(dest)
os.symlink(ckpt_path, dest)
print(f"✅ Checkpoint ready at {dest}")

In [4]:
!ls finetuning_data/manifests/

In [5]:
# Download manifests from your Hugging Face dataset and save locally (non-destructive)
import os, json, requests, random
from pathlib import Path

dataset_repo = "Firoj112/voxcpm-nepali-data"
desired_manifests = ["train_manifest.jsonl", "val_manifest.jsonl", "test_manifest.jsonl"]
base_url = f"https://huggingface.co/datasets/{dataset_repo}/resolve/main/manifests/"

manifests_dir = Path("finetuning_data/manifests")
manifests_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN")
headers = {"Authorization": f"Bearer {hf_token}"} if hf_token else {}

for fname in desired_manifests:
    out_path = manifests_dir / fname
    if out_path.exists():
        print(f"Already exists: {out_path}")
        continue
    # For validation try common alternative names on HF
    candidates = [fname]
    if fname == "val_manifest.jsonl":
        candidates = ["val_manifest.jsonl", "validation_manifest.jsonl", "validation_manifest.json"]
    downloaded = False
    for remote in candidates:
        url = base_url + remote
        print(f"Attempting to download {url}")
        try:
            r = requests.get(url, headers=headers, stream=True, timeout=60)
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f"Saved {out_path} ({out_path.stat().st_size} bytes)")
            downloaded = True
            break
        except requests.HTTPError as e:
            status = getattr(e.response, "status_code", None) if hasattr(e, "response") else None
            if status == 404:
                print(f"Not found: {url} (404). Trying next candidate.")
                continue
            else:
                print(f"Failed to download {url}: {e}")
        except Exception as e:
            print(f"Failed to download {url}: {e}")
    if not downloaded and fname == "val_manifest.jsonl":
        # fallback: create a small validation manifest from train manifest (5% up to 500 lines)
        train_path = manifests_dir / "train_manifest.jsonl"
        if train_path.exists():
            print("Validation manifest not found; creating small validation manifest from train_manifest.jsonl")
            with open(train_path, "r", encoding="utf-8") as tf:
                lines = tf.read().splitlines()
            if not lines:
                print("Train manifest is empty; cannot create validation manifest")
            else:
                n = max(1, int(len(lines) * 0.05))
                n = min(n, 500)
                rng = random.Random(42)
                sampled = rng.sample(lines, n) if len(lines) > n else lines[-n:]
                with open(out_path, "w", encoding="utf-8") as vf:
                    for line in sampled:
                        vf.write(line + "\n")
                print(f"Created {out_path} with {len(sampled)} lines")
        else:
            print("Cannot create val_manifest.jsonl because train_manifest.jsonl is missing locally. Please upload a validation manifest to HF or generate manifests locally.")

# Quick inspect top 3 lines of train manifest
train_path = manifests_dir / "train_manifest.jsonl"
if train_path.exists():
    with open(train_path, "r", encoding="utf-8") as f:
        for _ in range(3):
            line = f.readline()
            if not line:
                break
            try:
                obj = json.loads(line)
            except Exception:
                print("Failed to parse line as JSON:", line.strip()[:200])
                continue
            print({k: obj.get(k) for k in ["id", "audio", "text", "duration"]})

In [6]:
# Verify manifests exist and show line counts
from pathlib import Path
m = Path("finetuning_data/manifests")
if not m.exists():
    print("Manifest directory missing")
else:
    print("Manifests present:", [p.name for p in m.glob("*.jsonl")])
    for p in m.glob("*.jsonl"):
        size = p.stat().st_size
        with open(p, encoding='utf-8') as f:
            lines = sum(1 for _ in f)
        print(f"{p.name}: {lines} lines, {size} bytes")

In [7]:
# 🚀 START DISTRIBUTED TRAINING (T4 x 2)
# We use torchrun for distributed training across 2 GPUs
!torchrun --nproc_per_node=2 src/chatterbox/train_nepali.py \
    --manifest finetuning_data/manifests/train_manifest.jsonl \
    --resume_t3_weights t3_nepali_epoch_20.pt \
    --batch_size 16 \
    --epochs 50 \
    --lr 2e-5 \
    --save_every 5 \
    --num_workers 2 \
    --distributed

In [ ]:
# 📤 PUSH FINAL MODEL TO HUGGING FACE
from huggingface_hub import HfApi
import os

api = HfApi()
repo_id = "officialuser/chatterbox-nepali"
token = os.environ.get("HF_TOKEN")

if token and os.path.exists("t3_mtl_nepali_final.safetensors"):
    print(f"Pushing model to {repo_id}...")
    api.upload_file(
        path_or_fileobj="t3_mtl_nepali_final.safetensors",
        path_in_repo="t3_mtl_nepali_final.safetensors",
        repo_id=repo_id,
        repo_type="model",
        token=token
    )
    print("✅ Successfully pushed to Hugging Face!")
else:
    print("❌ HF_TOKEN missing or final model not found.")